# **UNIVERSIDADE FEDERAL DO CEARA**
---
Disciplina: Introducao a analise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Julio Cesar Gama Feitosa Freitas - 583956
2.   Vitoria Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 8 — EDA: 5 perguntas de negócio

## 🎯 Objetivo

Rodar as 5 análises exploratórias vistas no slide 18-19 do DIA 2, registrando o insight de negócio de cada uma — não só o número.


**Rota B - DuckDB + Python/Google Colab**

In [4]:
# Importa as bibliotecas e cria as pastas utilizadas pelo laboratorio
import os
import shutil
import duckdb

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)
os.makedirs("bigdata/bronze", exist_ok=True)
os.makedirs("bigdata/silver", exist_ok=True)

# Abre uma conexao DuckDB
con = duckdb.connect()

print("Ambiente preparado.")

Ambiente preparado.


## Reconstruir a Bronze e a Silver (mesmas regras do Lab 6)

In [5]:
# Faz o upload dos CSVs brutos: customers_synthetic.csv e transactions_synthetic.csv
# Nao envie fraud_labels.csv - ele nao e usado neste lab
#from google.colab import files

uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [6]:
# Copia os CSVs enviados para a estrutura Raw do projeto
# Usa "in name" em vez de igualdade exata para tolerar sufixos que o Colab
# adiciona automaticamente quando um arquivo com o mesmo nome ja existe,
# como "customers_synthetic (1).csv"
for name in uploaded:
    if "customers_synthetic" in name:
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    elif "transactions_synthetic" in name:
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw preparados.")

Arquivos Raw preparados.


In [7]:
# Recria a Bronze de clientes e de transacoes com as mesmas regras do Lab 6
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

con.sql(f"""
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id, name, cpf, email, segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
""")

# O CASE converte explicitamente True/False (texto) para BOOLEAN
con.sql(f"""
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id, customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type, status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE WHEN is_fraud = 'True' THEN true ELSE false END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
""")

con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()
con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [8]:
# Recria a Silver juntando transacoes com clientes e derivando as colunas de analise
con.sql("""
CREATE OR REPLACE TABLE silver_transactions AS
SELECT
  t.transaction_id, t.customer_id, t.amount, t.transaction_type,
  t.status, t.risk_score, t.is_fraud, t.ts,
  c.segment, c.credit_score,
  year(t.ts)  AS year,
  month(t.ts) AS month,
  day(t.ts)   AS day,
  dayofweek(t.ts) AS day_of_week,
  CASE
    WHEN t.amount < 100  THEN 'baixo'
    WHEN t.amount < 1000 THEN 'medio'
    ELSE 'alto'
  END AS amount_band
FROM bronze_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
""")

# Confere a quantidade de registros criada na Silver
con.sql("SELECT COUNT(*) FROM silver_transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       100000 │
└──────────────┘



In [9]:
# Salva a Bronze e a Silver em Parquet
con.sql("COPY bronze_customers TO 'bigdata/bronze/customers.parquet' (FORMAT PARQUET)")
con.sql("COPY bronze_transactions TO 'bigdata/bronze/transactions.parquet' (FORMAT PARQUET)")
con.sql("COPY silver_transactions TO 'bigdata/silver/transactions_enriched.parquet' (FORMAT PARQUET)")

print("Bronze e Silver salvas em bigdata/bronze/ e bigdata/silver/")

Bronze e Silver salvas em bigdata/bronze/ e bigdata/silver/


## 1️⃣ Segmentação por score

In [10]:
# Score medio de credito e quantidade de clientes distintos por segmento
con.sql("""
SELECT segment,
       ROUND(AVG(credit_score), 1) AS score_medio,
       COUNT(DISTINCT customer_id) AS clientes
FROM silver_transactions
GROUP BY segment
ORDER BY score_medio
""").show()

┌───────────┬─────────────┬──────────┐
│  segment  │ score_medio │ clientes │
│  varchar  │   double    │  int64   │
├───────────┼─────────────┼──────────┤
│ Standard  │       643.9 │     2665 │
│ Premium   │       652.7 │     5563 │
│ High-Risk │       659.8 │      876 │
└───────────┴─────────────┴──────────┘



**Registre:** qual segmento tem o menor score medio? Isso confirma a logica de risco do dataset?

*Standart*

## 2️⃣ Análise de risco por tipo de transação

In [11]:
# Quantidade de fraudes, total de transacoes e taxa de fraude por transaction_type
con.sql("""
SELECT transaction_type,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes,
       COUNT(*) AS total,
       ROUND(100.0*SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)/COUNT(*), 2) AS taxa_pct
FROM silver_transactions
GROUP BY transaction_type
ORDER BY taxa_pct DESC
""").show()

┌──────────────────┬─────────┬───────┬──────────┐
│ transaction_type │ fraudes │ total │ taxa_pct │
│     varchar      │ int128  │ int64 │  double  │
├──────────────────┼─────────┼───────┼──────────┤
│ transferencia    │     390 │ 20102 │     1.94 │
│ pagamento        │     196 │ 10180 │     1.93 │
│ compra           │     929 │ 49782 │     1.87 │
│ saque            │     318 │ 19936 │      1.6 │
└──────────────────┴─────────┴───────┴──────────┘



**Registre:** qual tipo de transacao concentra a maior taxa de fraude? Faz sentido priorizar verificacao extra nele?

*Transferencia. Não necessariamente. Apesar de ser a maior taxa, a quantidade de fraudes por compra é mais crítica, dado o alto volume de fraudes mesmo com taxa levemente menor que transferência.*

## 3️⃣ Padrões temporais

In [12]:
# Os 10 dias do mes com mais fraudes registradas, junto do volume total de transacoes
con.sql("""
SELECT day,
       COUNT(*) AS total_transacoes,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes
FROM silver_transactions
GROUP BY day
ORDER BY fraudes DESC
LIMIT 10
""").show()

┌───────┬──────────────────┬─────────┐
│  day  │ total_transacoes │ fraudes │
│ int64 │      int64       │ int128  │
├───────┼──────────────────┼─────────┤
│    21 │             7777 │     161 │
│    26 │             7913 │     157 │
│    24 │             7792 │     148 │
│    27 │             7801 │     144 │
│    20 │             7850 │     140 │
│    25 │             7689 │     140 │
│    23 │             7696 │     133 │
│    22 │             7838 │     132 │
│    28 │             7914 │     131 │
│    11 │             1505 │      41 │
└───────┴──────────────────┴─────────┘
  10 rows                  3 columns



**Registre:** os dias com mais fraude coincidem com os dias de mais volume (fim de mes)? Ou fraude tem padrao proprio?

*Não, as fraudes independem do total de transações. Os dias com maior quantidade de transações tem número total de fraudes menor que dias com menos transações.*

## 4️⃣ Cohort — clientes antigos vs novos

In [13]:
# Compara ticket medio e volume de transacoes entre clientes com mais e menos de 1 ano de casa
con.sql("""
SELECT
  CASE WHEN DATEDIFF('day', c.created_at, CURRENT_DATE) > 365 THEN 'cliente antigo'
       ELSE 'cliente novo' END AS cohort,
  ROUND(AVG(t.amount), 2) AS ticket_medio,
  COUNT(*) AS transacoes
FROM silver_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
GROUP BY 1
""").show()

┌────────────────┬──────────────┬────────────┐
│     cohort     │ ticket_medio │ transacoes │
│    varchar     │    double    │   int64    │
├────────────────┼──────────────┼────────────┤
│ cliente novo   │       184.99 │      44171 │
│ cliente antigo │       182.84 │      55829 │
└────────────────┴──────────────┴────────────┘



**Registre:** clientes antigos gastam mais por transacao? A diferenca e grande o suficiente para virar uma acao de negocio?

*Não, gastam menos. A diferença do ticket médio não é tão grande, mas o volume de transações pode justificar uma ação.*

## 5️⃣ Cross-sell — clientes Premium de alta frequência

In [14]:
# Clientes Premium com mais de 10 compras - candidatos a uma campanha de cross-sell direcionada
con.sql("""
SELECT customer_id, COUNT(*) AS compras
FROM silver_transactions
WHERE segment = 'Premium' AND transaction_type = 'compra'
GROUP BY customer_id
HAVING COUNT(*) > 10
ORDER BY compras DESC
LIMIT 10
""").show()

# Se a query acima vier vazia, o corte de > 10 pode ser exigente demais
# para uma amostra sintetica - descomente a linha abaixo para testar com > 5.
# con.sql("""
# SELECT customer_id, COUNT(*) AS compras
# FROM silver_transactions
# WHERE segment = 'Premium' AND transaction_type = 'compra'
# GROUP BY customer_id
# HAVING COUNT(*) > 5
# ORDER BY compras DESC
# LIMIT 10
# """).show()

┌─────────────┬─────────┐
│ customer_id │ compras │
│    int64    │  int64  │
├─────────────┼─────────┤
│        1920 │      49 │
│        8500 │      44 │
│        7531 │      41 │
│        5251 │      40 │
│        9306 │      40 │
│        5354 │      39 │
│        1072 │      39 │
│        7105 │      39 │
│        3686 │      38 │
│        4400 │      37 │
└─────────────┴─────────┘
  10 rows     2 columns



**Registre:** quantos clientes se qualificam? Isso e uma lista pequena o suficiente para uma campanha direcionada, ou grande demais?

*Apenas 10 clientes. É uma lista pequena suficiente para uma campanha direcionada*

## Consolidando os 5 achados (o framework do DIA 3)

Para cada pergunta acima, complete a tabela - isso vai direto para a apresentacao final:

| # | Raw Finding (o numero) | Business Insight (por que importa) | Acao proposta |
|---|------------------------|-------------------------------------|----------------|
| 1 | 643.9 |Standard contém o menor score dos 3 | |
| 2 | 390 | Transferência concentra a maior taxa de fraudes, mas não tão alta em relação aos demais | Focar em compras, pois tem um volume maior de fraudes |
| 3 | | As fraudes independem do total de transações | |
| 4 | 182.71 | Ticket médio de clientes antigos é menor | Propor uma ação para esses clientes, visto que o volume de usuários é mais expressivo |
| 5 | 10 | São poucos clientes que se qualificam | É possível direcionar uma ação para essa lista reduzida |

## Checkpoint

- [ ] `bronze_customers`, `bronze_transactions` e `silver_transactions` reconstruidas nesta sessao
- [ ] Parquets baixados para a maquina local (evita repetir o upload dos CSVs no proximo lab)
- [ ] As 5 queries rodaram sem erro
- [ ] Voce preencheu a tabela de Raw Finding -> Insight -> Acao para pelo menos 3 das 5
- [ ] Os numeros fazem sentido com o que ja vimos nos labs anteriores (fraude ~1,83%, High-Risk concentrando mais risco)

---

**Proximo lab:** `DIA3_LAB09_EXPORT.md` - tirar a Gold layer do Hive.